# nb126 — PXR docking ensemble (AutoDock Vina) on Kaggle

Dock every PXR train + test compound into PXR co-crystal structures, extract a binding affinity vector per compound across receptors.

Pipeline:
1. Pull PXR LBD co-crystal PDBs from RCSB (1ILH=SR12813, 2O9I=T0901317, 1NRL=PCN, 1M13=hyperforin, 4S0R, 5X0R)
2. Prepare receptors (strip waters/het, add hydrogens, convert to PDBQT via openbabel)
3. Generate 3D conformers via RDKit ETKDGv3
4. Dock each compound × each receptor with Vina
5. Save best-affinity matrix as PXR features

Vina is installed via apt or direct binary download from ccsb-scripps releases.

In [ ]:
import subprocess, sys, os, time, urllib.request, stat
from pathlib import Path
os.environ['PYTHONUNBUFFERED'] = '1'

# Install tools — try apt first, then download static binaries
print('[setup] installing tools...')
subprocess.run(['apt-get', 'update', '-q'], check=False, capture_output=True)
subprocess.run(['apt-get', 'install', '-y', '-q', 'autodock-vina', 'openbabel'], check=False, capture_output=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'rdkit', 'meeko'], check=False)

# openbabel
r = subprocess.run(['obabel', '-V'], capture_output=True, text=True)
print(f'openbabel: {(r.stdout + r.stderr).strip()[:80]}')

# Find or fetch vina
VINA_BIN = None
r = subprocess.run(['which', 'vina'], capture_output=True, text=True)
if r.stdout.strip():
    VINA_BIN = r.stdout.strip()
else:
    print('[setup] vina not via apt; downloading binary...')
    url = 'https://github.com/ccsb-scripps/AutoDock-Vina/releases/download/v1.2.5/vina_1.2.5_linux_x86_64'
    VINA_BIN = '/kaggle/working/vina'
    try:
        urllib.request.urlretrieve(url, VINA_BIN)
        os.chmod(VINA_BIN, os.stat(VINA_BIN).st_mode | stat.S_IEXEC)
    except Exception as e:
        print(f'binary download failed: {e}')
        VINA_BIN = None

if VINA_BIN:
    r = subprocess.run([VINA_BIN, '--version'], capture_output=True, text=True)
    out = (r.stdout + r.stderr).strip()[:120]
    print(f'vina: {out}')
else:
    raise RuntimeError('vina unavailable')

print(f'\nUsing VINA_BIN = {VINA_BIN}')

In [ ]:
# Download PXR PDB structures
PDB_DIR = Path('/kaggle/working/pdbs')
PDB_DIR.mkdir(exist_ok=True)

PXR_PDBS = ['1ILH', '2O9I', '1NRL', '1M13', '4S0R', '5X0R']
for pdb_id in PXR_PDBS:
    out = PDB_DIR / f'{pdb_id}.pdb'
    if not out.exists():
        try:
            urllib.request.urlretrieve(f'https://files.rcsb.org/download/{pdb_id}.pdb', out)
        except Exception as e:
            print(f'  {pdb_id}: download failed ({e})')
            continue
    print(f'  {pdb_id}: {out.stat().st_size//1024} KB')

In [ ]:
# Prepare receptors: strip waters, find co-crystal ligand center for pocket box
def prepare_receptor(pdb_path, out_pdbqt):
    lines = open(pdb_path).readlines()
    keep = []
    lig_coords = []
    for line in lines:
        if line.startswith('HETATM'):
            if 'HOH' in line or ' ZN ' in line or ' MG ' in line or ' NA ' in line:
                continue
            try:
                lig_coords.append((float(line[30:38]), float(line[38:46]), float(line[46:54])))
            except ValueError:
                pass
            continue
        if line.startswith(('ATOM', 'TER', 'END')):
            keep.append(line)
    clean_pdb = pdb_path.with_suffix('.clean.pdb')
    clean_pdb.write_text(''.join(keep))
    subprocess.run(['obabel', str(clean_pdb), '-O', str(out_pdbqt), '-xr'],
                   check=False, capture_output=True)
    if lig_coords:
        cx = sum(c[0] for c in lig_coords) / len(lig_coords)
        cy = sum(c[1] for c in lig_coords) / len(lig_coords)
        cz = sum(c[2] for c in lig_coords) / len(lig_coords)
    else:
        cx = cy = cz = 0.0
    return cx, cy, cz

receptors = {}
for pdb_id in PXR_PDBS:
    pdb = PDB_DIR / f'{pdb_id}.pdb'
    if not pdb.exists(): continue
    pdbqt = PDB_DIR / f'{pdb_id}.pdbqt'
    cx, cy, cz = prepare_receptor(pdb, pdbqt)
    size_kb = pdbqt.stat().st_size // 1024 if pdbqt.exists() else 0
    print(f'  {pdb_id}: center=({cx:.1f},{cy:.1f},{cz:.1f})  pdbqt={size_kb}KB')
    if pdbqt.exists() and size_kb > 0:
        receptors[pdb_id] = (str(pdbqt), cx, cy, cz)
print(f'\nUsable receptors: {len(receptors)}')

In [ ]:
# Load PXR train + test from HuggingFace (more reliable than Kaggle dataset path)
import pandas as pd, urllib.request
HF = 'https://huggingface.co/datasets/openadmet/pxr-challenge-train-test/resolve/main'
TR_LOC = '/kaggle/working/train.csv'
TE_LOC = '/kaggle/working/test.csv'
if not Path(TR_LOC).exists():
    urllib.request.urlretrieve(f'{HF}/pxr-challenge_TRAIN.csv', TR_LOC)
if not Path(TE_LOC).exists():
    urllib.request.urlretrieve(f'{HF}/pxr-challenge_TEST_BLINDED.csv', TE_LOC)
tr = pd.read_csv(TR_LOC)
te = pd.read_csv(TE_LOC)
tr.columns = [c.lower() for c in tr.columns]
te.columns = [c.lower() for c in te.columns]
name_col_tr = 'molecule name' if 'molecule name' in tr.columns else 'molecule_name'
name_col_te = 'molecule name' if 'molecule name' in te.columns else 'molecule_name'
tr_rec = tr[[name_col_tr, 'smiles']].rename(columns={name_col_tr: 'name'}).dropna()
te_rec = te[[name_col_te, 'smiles']].rename(columns={name_col_te: 'name'}).dropna()
print(f'Train: {len(tr_rec)}  Test: {len(te_rec)}')
print(f'Sample train SMILES: {tr_rec.iloc[0]["smiles"][:60]}')

In [ ]:
# Ligand prep — generate 3D conformer + convert to PDBQT
from rdkit import Chem
from rdkit.Chem import AllChem

LIG_DIR = Path('/kaggle/working/ligands')
LIG_DIR.mkdir(exist_ok=True)

def make_ligand_pdbqt(name, smiles, out_dir):
    safe = ''.join(c if c.isalnum() else '_' for c in str(name))
    mol = Chem.MolFromSmiles(smiles)
    if mol is None: return None
    mol = Chem.AddHs(mol)
    params = AllChem.ETKDGv3()
    params.randomSeed = 42
    cid = AllChem.EmbedMolecule(mol, params)
    if cid < 0: return None
    try:
        AllChem.MMFFOptimizeMolecule(mol, maxIters=100)
    except Exception: pass
    pdb_path = out_dir / f'{safe}.pdb'
    pdbqt_path = out_dir / f'{safe}.pdbqt'
    try:
        Chem.MolToPDBFile(mol, str(pdb_path))
    except Exception: return None
    subprocess.run(['obabel', str(pdb_path), '-O', str(pdbqt_path), '-h'],
                   check=False, capture_output=True)
    return pdbqt_path if pdbqt_path.exists() and pdbqt_path.stat().st_size > 100 else None

demo = make_ligand_pdbqt(tr_rec.iloc[0]['name'], tr_rec.iloc[0]['smiles'], LIG_DIR)
print(f'demo: {demo}')

In [ ]:
# Dock function
import re
BOX_SIZE = 24.0
AFFINITY_RE = re.compile(r'\s*1\s+(-?\d+\.\d+)')

def dock_one(args):
    cname, smi, rec_id, rec_pdbqt, cx, cy, cz = args
    lig = make_ligand_pdbqt(cname, smi, LIG_DIR)
    if lig is None: return (cname, rec_id, None)
    safe = ''.join(c if c.isalnum() else '_' for c in str(cname))
    out = LIG_DIR / f'{safe}_{rec_id}.out.pdbqt'
    cmd = [VINA_BIN, '--receptor', str(rec_pdbqt), '--ligand', str(lig),
           '--center_x', str(cx), '--center_y', str(cy), '--center_z', str(cz),
           '--size_x', str(BOX_SIZE), '--size_y', str(BOX_SIZE), '--size_z', str(BOX_SIZE),
           '--num_modes', '1', '--exhaustiveness', '4',
           '--out', str(out)]
    try:
        r = subprocess.run(cmd, capture_output=True, text=True, timeout=120)
    except subprocess.TimeoutExpired:
        return (cname, rec_id, None)
    aff = None
    for line in r.stdout.splitlines():
        m = AFFINITY_RE.match(line)
        if m:
            aff = float(m.group(1)); break
    out.unlink(missing_ok=True)
    lig.unlink(missing_ok=True)
    return (cname, rec_id, aff)

# quick sanity check on 1 dock
rec_id0 = list(receptors.keys())[0]
pdbqt0, cx0, cy0, cz0 = receptors[rec_id0]
demo_res = dock_one((tr_rec.iloc[0]['name'], tr_rec.iloc[0]['smiles'], rec_id0, pdbqt0, cx0, cy0, cz0))
print(f'sanity: {demo_res}')

In [ ]:
# Build full task list
all_compounds = pd.concat([
    tr_rec.assign(split='train'),
    te_rec.assign(split='test'),
], ignore_index=True)
print(f'Docking {len(all_compounds)} compounds x {len(receptors)} receptors = {len(all_compounds)*len(receptors)} runs')

tasks = []
for _, row in all_compounds.iterrows():
    for rec_id, (pdbqt, cx, cy, cz) in receptors.items():
        tasks.append((row['name'], row['smiles'], rec_id, pdbqt, cx, cy, cz))
print(f'Tasks queued: {len(tasks)}')

In [ ]:
# Parallel dock — use all 4 cores; chunk output saves every 500 results
from concurrent.futures import ProcessPoolExecutor
import pickle

OUT_DIR = Path('/kaggle/working/dock_partials')
OUT_DIR.mkdir(exist_ok=True)

t0 = time.time()
results = []
with ProcessPoolExecutor(max_workers=4) as ex:
    for i, res in enumerate(ex.map(dock_one, tasks, chunksize=20)):
        results.append(res)
        if (i+1) % 200 == 0:
            elapsed = time.time() - t0
            eta_h = elapsed / (i+1) * (len(tasks) - i - 1) / 3600
            print(f'  {i+1}/{len(tasks)}  elapsed {elapsed/60:.1f}m  ETA {eta_h:.2f}h')
            # save partial every 200
            with open(OUT_DIR / f'partial_{i+1}.pkl', 'wb') as f:
                pickle.dump(results, f)

df_dock = pd.DataFrame(results, columns=['name', 'receptor', 'affinity'])
df_dock.to_parquet('/kaggle/working/docking_results.parquet', index=False)
print(f'\nDone. {len(df_dock)} dock results in {(time.time()-t0)/60:.1f}m')

In [ ]:
# Pivot to wide feature matrix
wide = df_dock.pivot_table(index='name', columns='receptor', values='affinity', aggfunc='min')
wide['best_affinity'] = wide.min(axis=1)
wide['mean_affinity'] = wide[list(receptors.keys())].mean(axis=1)
wide['std_affinity']  = wide[list(receptors.keys())].std(axis=1)
wide.reset_index().to_parquet('/kaggle/working/docking_features_wide.parquet', index=False)
print('Wide feature shape:', wide.shape)
print(wide.describe())